# MBE Dynamics in a Rollout
Inspect how Matrix-Based Entropy (MBE) evolves as the model generates tokens.
For each hidden layer, we compute MBE on prefixes of increasing length,
revealing how representation diversity builds up during chain-of-thought.

**Key questions:**
1. Does MBE increase monotonically as more tokens are generated?
2. Which layers show the sharpest MBE dynamics?
3. Do correct vs incorrect rollouts differ in their MBE trajectory?

In [ ]:
import os, sys, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath("."))
from script.analyze_rollouts import (
    DATASET_REGISTRY,
    run_dataset_rollouts,
    interpolate_mbe,
    compute_growth_profile,
    compute_mbe_velocity,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load Model & Data
Load a Qwen3 model and a few GSM8K examples. Adjust `MODEL_NAME` to point at a checkpoint if available.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"   # swap for a checkpoint path if available
MAX_NEW_TOKENS = 512
N_SAMPLES = 20                    # per dataset
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Datasets to analyze — any subset of DATASET_REGISTRY keys:
# "gsm8k" (math), "humaneval" (coding), "arc_challenge" (reasoning), "mmlu" (knowledge)
DATASETS = ["gsm8k", "arc_challenge", "humaneval", "mmlu"]

USE_VLLM = True
VLLM_GPU_MEMORY_UTILIZATION = 0.5

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B  |  Layers: 28  |  Device: cpu


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test set: 1319 examples


## 2. Helper Functions
Generate a rollout and extract hidden states, then compute MBE on prefixes of increasing length using `OnlineMBE`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map=DEVICE)
model.eval()
print(f"Model: {MODEL_NAME}  |  Layers: {model.config.num_hidden_layers}  |  Device: {DEVICE}")

vllm_engine = None
if USE_VLLM:
    try:
        from vllm import LLM
        vllm_engine = LLM(
            model=MODEL_NAME,
            gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
            dtype="bfloat16",
            max_model_len=MAX_NEW_TOKENS + 512,
        )
        print(f"vLLM engine loaded ({VLLM_GPU_MEMORY_UTILIZATION:.0%} VRAM)")
    except Exception as e:
        print(f"vLLM unavailable ({e}), falling back to HF generate")

Helpers ready.


## 3. Run Rollouts
Generate completions for a few GSM8K problems and compute per-layer MBE dynamics.

In [ ]:
all_results = {}  # dataset_name -> list of result dicts

for ds in DATASETS:
    print(f"\n=== {ds} ({DATASET_REGISTRY[ds]['category']}) ===")
    results = run_dataset_rollouts(
        model, tokenizer, ds, N_SAMPLES,
        vllm_engine=vllm_engine, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
    )
    all_results[ds] = results
    n_correct = sum(r["correct"] for r in results)
    vels = [r["mbe_velocity"] for r in results]
    vel_c = np.mean([r["mbe_velocity"] for r in results if r["correct"]] or [float("nan")])
    vel_i = np.mean([r["mbe_velocity"] for r in results if not r["correct"]] or [float("nan")])
    print(f"  acc={n_correct}/{len(results)} ({n_correct/len(results):.0%})  "
          f"vel_all={np.mean(vels):.4f}  vel_correct={vel_c:.4f}  vel_incorrect={vel_i:.4f}")

[1/6] ✗  pred=          gold=      33  len=512
[2/6] ✗  pred=    3.00  gold=      57  len=512
[3/6] ✗  pred=       3  gold=       7  len=512
[4/6] ✗  pred=          gold=      48  len=512
[5/6] ✗  pred=      16  gold=    5600  len=512
[6/6] ✗  pred=          gold=      60  len=512

Correct: 0  |  Incorrect: 6


## 3. MBE Velocity × Correctness Correlation
For each dataset, compare mean MBE velocity (mean ΔMBE, last layer) between correct and incorrect rollouts.
A positive delta means correct rollouts have faster-growing representation diversity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

ds_names, vel_correct, vel_incorrect, correlations = [], [], [], []
for ds, results in all_results.items():
    c = [r["mbe_velocity"] for r in results if r["correct"]]
    i = [r["mbe_velocity"] for r in results if not r["correct"]]
    vels = np.array([r["mbe_velocity"] for r in results])
    labels = np.array([int(r["correct"]) for r in results])
    if vels.std() > 0 and labels.std() > 0:
        corr = float(np.corrcoef(vels, labels)[0, 1])
    else:
        corr = 0.0
    ds_names.append(f"{ds}\n({DATASET_REGISTRY[ds]['category']})")
    vel_correct.append(np.mean(c) if c else float("nan"))
    vel_incorrect.append(np.mean(i) if i else float("nan"))
    correlations.append(corr)

x = np.arange(len(ds_names))
w = 0.35
ax = axes[0]
ax.bar(x - w/2, vel_correct,  w, label="Correct",   color="#27ae60", alpha=0.85)
ax.bar(x + w/2, vel_incorrect, w, label="Incorrect", color="#e74c3c", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ds_names, fontsize=9)
ax.set_ylabel("Mean MBE velocity (ΔMBE/token)"); ax.set_title("MBE Velocity: Correct vs Incorrect")
ax.legend(); ax.grid(axis="y", alpha=0.3)

ax = axes[1]
colors = ["#27ae60" if c > 0 else "#e74c3c" for c in correlations]
ax.bar(x, correlations, color=colors, alpha=0.85)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(ds_names, fontsize=9)
ax.set_ylabel("Pearson r (MBE velocity vs correctness)")
ax.set_title("Correlation: MBE Velocity × Rollout Quality")
ax.grid(axis="y", alpha=0.3)

fig.suptitle("MBE Velocity vs Rollout Quality — Multi-Dataset", fontsize=13)
plt.show()
print("Pearson r per dataset:", dict(zip([d.split("\n")[0] for d in ds_names], [f"{c:+.3f}" for c in correlations])))

KeyError: 'full_mbe'

## 4. Per-dataset Averaged MBE Trajectories
Interpolate each rollout to a common 0–100% completion axis and average correct vs incorrect.

In [ ]:
N_POINTS = 50
common_axis = np.linspace(0, 100, N_POINTS)

n_datasets = len(all_results)
fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), constrained_layout=True)
if n_datasets == 1:
    axes = [axes]

for ax, (ds, results) in zip(axes, all_results.items()):
    n_layers = results[0]["mbe"].shape[0]
    layer = n_layers - 1  # last layer

    correct_interp   = [interpolate_mbe(r["mbe"], N_POINTS) for r in results if r["correct"]]
    incorrect_interp = [interpolate_mbe(r["mbe"], N_POINTS) for r in results if not r["correct"]]
    correct_interp   = [x for x in correct_interp if x is not None]
    incorrect_interp = [x for x in incorrect_interp if x is not None]

    for group, color, label in [(correct_interp, "#27ae60", "Correct"), (incorrect_interp, "#e74c3c", "Incorrect")]:
        if not group:
            continue
        mean = np.mean([g[layer] for g in group], axis=0)
        std  = np.std( [g[layer] for g in group], axis=0) if len(group) > 1 else np.zeros(N_POINTS)
        ax.plot(common_axis, mean, color=color, linewidth=2, label=f"{label} (n={len(group)})")
        ax.fill_between(common_axis, mean - std, mean + std, color=color, alpha=0.15)

    ax.set_title(f"{ds}\n({DATASET_REGISTRY[ds]['category']})", fontsize=10)
    ax.set_xlabel("Completion %"); ax.grid(True, alpha=0.3)
    if ax is axes[0]:
        ax.set_ylabel("Cumulative MBE (last layer)")
    ax.legend(fontsize=7)

fig.suptitle("Averaged MBE Trajectory: Correct vs Incorrect", fontsize=13)
plt.show()

## 5. Single-rollout Inspection
Heatmap and token-aligned MBE for a selected rollout. Adjust `INSPECT_DS` and `INSPECT_IDX`.

In [ ]:
INSPECT_DS  = DATASETS[0]   # which dataset to inspect
INSPECT_IDX = 0             # which rollout index within that dataset

r = all_results[INSPECT_DS][INSPECT_IDX]
mbe = r["mbe"]
n_layers, comp_len = mbe.shape
tag = "✓" if r["correct"] else "✗"

fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
im = ax.imshow(mbe, aspect="auto", cmap="YlOrRd", origin="lower")
ax.set_xlabel("Token position"); ax.set_ylabel("Layer")
ax.set_title(f"{INSPECT_DS} [{INSPECT_IDX}]  {tag}  gold={r['gold']}  pred={r.get('predicted','?')}  len={comp_len}")
plt.colorbar(im, ax=ax, shrink=0.8, label="Cumulative MBE")
plt.show()

## 6. MBE Growth Profile
Per-layer total MBE growth and half-life for correct vs incorrect rollouts.

In [ ]:
n_datasets = len(all_results)
fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), constrained_layout=True)
if n_datasets == 1:
    axes = [axes]

for ax, (ds, results) in zip(axes, all_results.items()):
    n_layers = results[0]["mbe"].shape[0]
    layers_arr = np.arange(1, n_layers + 1)

    for group, color, label in [
        ([r for r in results if r["correct"]],     "#27ae60", "Correct"),
        ([r for r in results if not r["correct"]], "#e74c3c", "Incorrect"),
    ]:
        valid = [r for r in group if r["comp_len"] > 1]
        if not valid:
            continue
        profiles = [compute_growth_profile(r["mbe"]) for r in valid]
        g_mean = np.mean([p[0] for p in profiles], axis=0)
        ax.plot(layers_arr, g_mean, color=color, linewidth=2, label=f"{label} (n={len(valid)})",
                marker="o", markersize=3)

    ax.set_title(f"{ds}\n({DATASET_REGISTRY[ds]['category']})", fontsize=10)
    ax.set_xlabel("Layer"); ax.grid(True, alpha=0.3)
    if ax is axes[0]:
        ax.set_ylabel("MBE growth (final − initial)")
    ax.legend(fontsize=7)

fig.suptitle("Per-layer MBE Growth: Correct vs Incorrect", fontsize=13)
plt.show()

In [ ]:
## 7. Text-Aligned MBE
Token strip with ΔMBE colour-coding — useful for qualitative inspection of what events drive MBE jumps.

In [ ]:
def plot_text_mbe(result, layer_idx=-1, window=80, offset=0):
    """Token strip + MBE line plot for a single rollout."""
    tokens  = result["tokens"]
    mbe     = result["mbe"]
    n_layers = mbe.shape[0]
    if layer_idx < 0:
        layer_idx = n_layers + layer_idx
    comp_len = len(tokens)
    end = min(offset + window, comp_len)
    tok_slice = tokens[offset:end]
    mbe_slice = mbe[layer_idx, offset:end]
    delta_full = np.concatenate([[0.0], np.diff(mbe[layer_idx])])
    delta_slice = delta_full[offset:end]

    fig = plt.figure(figsize=(18, 5))
    gs  = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)
    ax_line = fig.add_subplot(gs[0])
    ax_text = fig.add_subplot(gs[1], sharex=ax_line)
    x = np.arange(len(tok_slice))

    ax_line.plot(x, mbe_slice, color="#2c3e50", linewidth=1.5)
    ax_line.fill_between(x, mbe_slice, alpha=0.12, color="#3498db")
    threshold = np.percentile(np.abs(delta_slice), 90) if len(delta_slice) else 0
    for j, d in enumerate(delta_slice):
        if np.abs(d) > threshold:
            ax_line.axvline(j, color="#e74c3c" if d > 0 else "#3498db", alpha=0.3, linewidth=1)
    tag = "✓" if result["correct"] else "✗"
    ax_line.set_ylabel(f"MBE layer {layer_idx+1}")
    ax_line.set_title(f"{result.get('dataset','?')} [{tag}]  tokens {offset}–{end}  "
                      f"gold={result['gold']}  vel={result['mbe_velocity']:.4f}", fontsize=10)
    ax_line.grid(True, alpha=0.2)
    plt.setp(ax_line.get_xticklabels(), visible=False)

    abs_max = max(np.abs(delta_slice).max(), 1e-8)
    norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
    cmap = plt.cm.RdBu_r
    for i, (tok, d) in enumerate(zip(tok_slice, delta_slice)):
        ax_text.axvspan(i - 0.5, i + 0.5, color=cmap(norm(d)), alpha=0.7)
        disp = tok.replace("\n", "↵").replace("\t", "→")[:5]
        ax_text.text(i, 0.5, disp, ha="center", va="center", fontsize=5.5,
                     fontfamily="monospace", rotation=90)
    ax_text.set_xlim(-0.5, len(tok_slice) - 0.5)
    ax_text.set_ylim(0, 1); ax_text.set_yticks([])
    ax_text.set_xlabel("Token position (relative to completion start)")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax_text, orientation="vertical", shrink=0.8, pad=0.02, label="ΔMBE")
    fig.tight_layout()
    return fig

print("plot_text_mbe ready.")

r = all_results[INSPECT_DS][INSPECT_IDX]
plot_text_mbe(r, layer_idx=-1, window=80, offset=0)
plt.show()

In [ ]:
# Browse in 80-token windows — adjust offset to page through the rollout
plot_text_mbe(r, layer_idx=-1, window=80, offset=80)
plt.show()

# Compare early vs late layers for the same rollout
n_layers = r["mbe"].shape[0]
for layer in [0, n_layers // 2, n_layers - 1]:
    plot_text_mbe(r, layer_idx=layer, window=80, offset=0)
plt.show()

In [ ]:
# (unused — kept for notebook cell count compatibility)

In [ ]:
# (unused — kept for notebook cell count compatibility)

In [ ]:
# (unused — kept for notebook cell count compatibility)

In [ ]:
# (unused — kept for notebook cell count compatibility)

In [ ]:
# (unused — kept for notebook cell count compatibility)

In [ ]:
# (unused — kept for notebook cell count compatibility)